In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import os
import os.path as osp
import glob
from typing import Any, Dict, List
from torch.utils.data import Dataset, DataLoader
from Net.dataset import NCLTDataset, nclt_collate
from Net.models import KalmanNet
from Net.utils import MODELS, MAE, MSE, MSEdB
from mmengine import Config
import tensorflow as tf

I0000 00:00:1779701778.466791  136307 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
dataset = NCLTDataset('./data/NCLT/processed/test.pt')
loader = DataLoader(dataset, collate_fn=nclt_collate, batch_size=128)
for batch in loader:
    break

In [3]:
print(f"Total sequences in test set: {len(dataset)}")

Total sequences in test set: 3


In [4]:
# Find latest checkpoint and config automatically
runs_dir = './runs/new_exp'
if not osp.exists(runs_dir):
    # Fallback to weights folder if runs not found
    weight_path = 'weights/kalman560/epoch=44-val_loss=75.70-val_MSE_dB=18.79.ckpt'
    cfg_path = 'weights/kalman560/config.py'
    print("Using hardcoded fallback weights.")
else:
    # Find newest version folder
    version_dirs = sorted(glob.glob(osp.join(runs_dir, 'Wheel_GPS_*_v*')), key=osp.getmtime, reverse=True)
    latest_run = version_dirs[0]
    
    # Find newest checkpoint
    ckpt_files = sorted(glob.glob(osp.join(latest_run, 'checkpoints', '*.ckpt')), key=osp.getmtime, reverse=True)
    weight_path = ckpt_files[0]
    
    # Find config
    cfg_path = osp.join(latest_run, 'configs', 'config.py')
    
    print(f"Latest Run: {latest_run}")
    print(f"Latest Checkpoint: {weight_path}")
    print(f"Latest Config: {cfg_path}")

config = Config.fromfile(cfg_path)

Latest Run: ./runs/new_exp/Wheel_GPS_l50w4step2_origin_v26
Latest Checkpoint: ./runs/new_exp/Wheel_GPS_l50w4step2_origin_v26/checkpoints/epoch=49-val_loss=250125631488.00-val_MSE_dB=113.98.ckpt
Latest Config: ./runs/new_exp/Wheel_GPS_l50w4step2_origin_v26/configs/config.py


In [5]:
# Option to use ONNX or TFLite model instead of .ckpt
USE_ONNX = False
USE_TFLITE = True

onnx_path = osp.join(osp.dirname(weight_path), 'model_simp.onnx')
tflite_path = osp.join(osp.dirname(weight_path), 'model.tflite')

if USE_TFLITE and osp.exists(tflite_path):
    print(f"Using TFLite model: {tflite_path}")
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
elif USE_ONNX and osp.exists(onnx_path):
    import onnxruntime as ort
    print(f"Loading ONNX model from: {onnx_path}")
    ort_session = ort.InferenceSession(onnx_path)
else:
    model = MODELS.build(dict(type = config.trainer.type, cfg = config, save_dir = {}))
    print("Built model from checkpoint.")

Using TFLite model: ./runs/new_exp/Wheel_GPS_l50w4step2_origin_v26/checkpoints/model.tflite


/usr/lib/python3.14/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [6]:
from lightning import Trainer
from tqdm import tqdm

if USE_TFLITE:
    print("Running TFLite Inference...")
    predictions = []
    sensor_key = config.model.params.get('sensor_based', 'imu')
    
    # Map input names to indices (cleaner mapping)
    input_indices = {d['name']: d['index'] for d in input_details}
    
    for batch in tqdm(loader):
        # Batch contains [bs, dim, seq_len]
        batch_size, _, seq_len = batch[sensor_key].shape
        batch_preds = torch.zeros(batch_size, config.params.state_dim, seq_len)
        
        for b in range(batch_size):
            for t in range(seq_len):
                # Prepare inputs: [1, Features, 1] -> [1, 1, Features]
                sensor_input = batch[sensor_key][b:b+1, :, t:t+1].permute(0, 2, 1).numpy().astype(np.float32)
                correction_input = batch['filtered_gps'][b:b+1, :, t:t+1].permute(0, 2, 1).numpy().astype(np.float32)
                
                # Handle NaNs in correction
                correction_input = np.nan_to_num(correction_input)
                
                interpreter.set_tensor(input_indices['sensor'], sensor_input)
                interpreter.set_tensor(input_indices['correction'], correction_input)
                
                interpreter.invoke()
                
                # Output is also likely transposed: [1, 1, 6]
                out = interpreter.get_tensor(output_details[0]['index'])
                if out.ndim == 3 and out.shape[1] == 1:
                    out = np.transpose(out, (0, 2, 1))
                    
                batch_preds[b, :, t:t+1] = torch.from_numpy(out)
        
        res = {
            'preds': batch_preds, 
            'targets': batch['ground_truth'], 
            'mask': batch['mask'], 
            'data_date': batch['data_date'],
            'ground_truth': batch['ground_truth'],
            'gps': batch['gps']
        }
        predictions.append(res)
        
elif not USE_ONNX:
    trainer = Trainer(accelerator='cpu', devices=1)
    predictions = trainer.predict(
        model,
        loader,
        ckpt_path=weight_path
    )
else:
    # ONNX inference loop
    pass

Running TFLite Inference...


  0%|                                                                                                                                                                                                                                                   | 0/1 [00:00<?, ?it/s]


RuntimeError: Encountered unresolved custom op: ONNX_MAX.
See instructions: https://www.tensorflow.org/lite/guide/ops_custom Node number 37 (ONNX_MAX) failed to prepare.

In [ ]:
collect = {}
for preds in predictions:
    batch_date: list = preds['data_date']
    for idx, date in enumerate(batch_date):
        if date not in collect :
            collect[date] = {k: [v[idx]] for k, v in preds.items() if isinstance(v, (list, torch.Tensor))}
        else:
            for k, v in preds.items():
                if isinstance(v, (list, torch.Tensor)):
                    collect[date][k].append(v[idx])

In [ ]:
print(f"Evaluating dates: {list(collect.keys())}")

In [ ]:
# Multi-metric evaluation (RMSE, MAE, MSEdB)
rmse_metric = MSE(squared=False)
mae_metric = MAE()
msedb_metric = MSEdB()

times = ['2012-11-04','2012-11-16','2013-04-05']
all_stats = []

for time in times:
    if time not in collect: continue
    
    est = torch.hstack(collect[time]['preds'])
    mask = torch.hstack(collect[time]['mask'])
    gt = torch.hstack(collect[time]['ground_truth'])
    gps = torch.hstack(collect[time]['gps'])
    
    pred_masked = est[..., mask]
    gt_masked = gt[..., mask]
    gps_masked = gps[..., mask]
    
    rmse = rmse_metric(gt_masked, pred_masked)
    mae = mae_metric(gt_masked, pred_masked)
    msedb = msedb_metric(gt_masked, pred_masked)
    
    gps_rmse = rmse_metric(gt_masked, gps_masked)
    
    print(f"\n>>> {time} stats:")
    print(f"    Track Len: {gt_masked.shape[-1]}")
    print(f"    RMSE:      {rmse:.4f} m")
    print(f"    MAE:       {mae:.4f} m")
    print(f"    MSE:       {msedb:.4f} dB")
    print(f"    GPS RMSE:  {gps_rmse:.4f} m")
    
    all_stats.append({'rmse': rmse, 'mae': mae, 'msedb': msedb})

print("\n--- Overall Average ---")
if all_stats:
    avg_rmse = torch.mean(torch.stack([s['rmse'] for s in all_stats]))
    avg_mae = torch.mean(torch.stack([s['mae'] for s in all_stats]))
    print(f"AVG RMSE: {avg_rmse:.4f} m")
    print(f"AVG MAE:  {avg_mae:.4f} m")

In [7]:
def plot_path(est, gt, title='Path Comparison'):
    plt.figure(figsize=(12, 8))
    plt.plot(est[1,:], est[0,:], c='b', label='Estimate')
    plt.scatter(gt[1, :], gt[0, :], c='r', s=1, label='Ground Truth')
    plt.axis('equal')
    plt.legend()
    plt.title(title)
    plt.xlabel('East [m]')
    plt.ylabel('North [m]')
    plt.grid(True)
    plt.show()

In [8]:
time = '2012-11-16'
if time in collect:
    est = torch.hstack(collect[time]['preds'])
    mask = torch.hstack(collect[time]['mask'])
    gt = torch.hstack(collect[time]['ground_truth'])
    plot_path(est[...,mask], gt[...,mask], f"Trajectory for {time}")

NameError: name 'collect' is not defined

In [9]:
from lxml import etree
from data.NCLT.preprocess import _format_lat_lon, local_to_gps_coord
def export_to_kml(x1: list, y1:list, label1:str='test', dataset_date:str='test', output_folder='output'):
    template_path = './QGIS/template.kml'
    if not osp.exists(template_path): 
        print("KML Template missing!")
        return
        
    root = etree.parse(template_path).getroot()
    ns = {None : 'http://www.opengis.net/kml/2.2'}
    
    tags = root.findall('.//name', ns)
    tags[1].text = label1
    
    tags = root.findall('.//coordinates', ns)
    lat1, lon1 = local_to_gps_coord(x1, y1)
    tags[0].text = _format_lat_lon(lat1, lon1)
    
    out_path = f"./{output_folder}/{dataset_date}_{label1}.kml"
    os.makedirs(osp.dirname(out_path), exist_ok=True)
    with open(out_path, 'wb') as f:
        f.write(etree.tostring(root, xml_declaration=True, encoding='UTF-8', pretty_print=True))
    print(f"Exported KML to: {out_path}")